**Librerias**

In [41]:
import scipy.io as sio
import numpy as np
import pandas as pd

**Archivos**

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import zipfile
import os

ruta_zip = "/content/drive/MyDrive/datos_senales_datos_parkinson_cursos.zip"
ruta_extract = "/content/datos_eeg"

with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
    zip_ref.extractall(ruta_extract)

print("Archivos extraídos en:", ruta_extract)
print(os.listdir(ruta_extract))

Archivos extraídos en: /content/datos_eeg
['parkinson', 'control']


**Tipo de archivos**

In [44]:
ruta_control = os.path.join(ruta_extract, "control")
ruta_parkinson = os.path.join(ruta_extract, "parkinson")

print("Control:", os.listdir(ruta_control)[:5])
print("Parkinson:", os.listdir(ruta_parkinson)[:5])

Control: ['C004_EP_reposo.mat', 'C027_EP_reposo.mat', 'C021_EP_reposo.mat', 'C006_EP_reposo.mat', 'C040_EP_reposo.mat']
Parkinson: ['P016_EP_reposo.mat', 'P025_EP_reposo.mat', 'P049_EP_reposo.mat', 'P020_EP_reposo.mat', 'P012_EP_reposo.mat']


In [45]:
import scipy.io

def cargar_senal(ruta):
    data = scipy.io.loadmat(ruta)
    return data

# **Punto 1**


In [46]:
def energia_promedio(senal):
    if senal.ndim == 3:
        return np.mean(senal**2, axis=(1,2))
    elif senal.ndim == 2:
        return np.mean(senal**2, axis=1)
    else:
        raise ValueError("Dimensión no soportada")

Se emplea la energía promedio normalizada $E=\frac{1}{N}\sum x^2$ porque permite comparar de forma justa las señales EEG entre distintos sujetos y condiciones, independientemente del número de muestras o de épocas que tenga cada registro. Al dividir por el total de datos, se elimina el sesgo que introduce la duración de la señal, ya que señales más largas tienden a tener mayor energía total, obteniendo así una medida proporcional a la potencia promedio del canal.


Se selecciona un archivo del grupo control, se carga la señal EEG y se calcula la energía promedio por canal, obteniendo un vector que representa la actividad energética de cada canal

**Control**

In [48]:
# tomar un archivo del grupo control
archivo = os.listdir(ruta_control)[0]
ruta = os.path.join(ruta_control, archivo)

print("Archivo usado:", archivo)

# cargar señal
data = cargar_senal(ruta)
senal = data['data']

# calcular energía promedio por canal
energia = energia_promedio(senal)

print("Shape de la señal:", senal.shape)
print("Energía por canal:", energia)
print("Número de canales:", len(energia))

Archivo usado: C004_EP_reposo.mat
Shape de la señal: (8, 2000, 146)
Energía por canal: [ 7.07433666  9.14199983 14.37496607  7.13536346 14.39372299  7.33070887
  7.97007705  9.74994933]
Número de canales: 8


In [49]:
df_energia = pd.DataFrame(energia, columns=["Energía"])
df_energia.index.name = "Canal"

print(df_energia)

         Energía
Canal           
0       7.074337
1       9.142000
2      14.374966
3       7.135363
4      14.393723
5       7.330709
6       7.970077
7       9.749949


**Parkinson**

In [52]:
# tomar un archivo del grupo parkinson
archivop = os.listdir(ruta_parkinson)[0]
rutap = os.path.join(ruta_parkinson, archivop)

print("Archivo usado:", archivop)

# cargar señal
datap = cargar_senal(rutap)
senalp = datap['data']

# calcular energía promedio por canal
energiap = energia_promedio(senalp)

print("Shape de la señal:", senalp.shape)
print("Energía por canal:", energiap)
print("Número de canales:", len(energiap))

Archivo usado: P016_EP_reposo.mat
Shape de la señal: (8, 2000, 150)
Energía por canal: [ 6.07861491  6.69932926  8.83443883  7.42055235  5.64887112 19.3508238
 20.91448696 30.6643055 ]
Número de canales: 8


In [51]:
df_energiap = pd.DataFrame(energiap, columns=["Energía"])
df_energiap.index.name = "Canalp"

print(df_energiap)

          Energía
Canalp           
0        6.078615
1        6.699329
2        8.834439
3        7.420552
4        5.648871
5       19.350824
6       20.914487
7       30.664305


**Conclusión**
Se observa que el grupo de pacientes con Parkinson presenta valores de energía significativamente mayores en comparación con el grupo control, así como una mayor variabilidad entre canales. Esta diferencia podría estar asociada a alteraciones en la actividad neuronal características de la enfermedad. Sin embargo, la diferencia de magnitud también sugiere posibles variaciones en la escala o preprocesamiento de las señales, por lo que es importante validar la normalización de los datos.

# **Punto 2**

Se recorren todos los archivos de cada grupo (control y Parkinson), se calcula la energía promedio por canal para cada sujeto y se almacenan los resultados en listas, donde cada fila representa un sujeto y cada columna un canal.

In [55]:
energia_control = []
energia_parkinson = []

# CONTROL
for archivo in os.listdir(ruta_control):
    ruta1 = os.path.join(ruta_control, archivo)

    data = cargar_senal(ruta1)
    senal = data['data']

    energia = energia_promedio(senal)
    energia_control.append(energia)


# PARKINSON
for archivo in os.listdir(ruta_parkinson):
    ruta2 = os.path.join(ruta_parkinson, archivo)

    data2 = cargar_senal(ruta2)
    senal2p = data2['data']

    energia2p = energia_promedio(senal2p)
    energia_parkinson.append(energia2p)

In [56]:
# DataFrames
df_control = pd.DataFrame(energia_control)
df_parkinson = pd.DataFrame(energia_parkinson)

Los valores de energía por canal para cada sujeto se organizaron en DataFrames, donde cada fila representa un sujeto y cada columna un canal, permitiendo así realizar comparaciones estadísticas entre grupos

In [57]:
# Nombrar columnas como canales
df_control.columns = [f"Canal_{i}" for i in range(df_control.shape[1])]
df_parkinson.columns = df_control.columns

# Nombrar filas como sujetos
df_control.index = [f"Sujeto_C_{i}" for i in range(df_control.shape[0])]
df_parkinson.index = [f"Sujeto_P_{i}" for i in range(df_parkinson.shape[0])]

print("=== CONTROL ===")
print(df_control.head())

print("\n=== PARKINSON ===")
print(df_parkinson.head())

print("\nDimensiones:")
print("Control:", df_control.shape)
print("Parkinson:", df_parkinson.shape)

=== CONTROL ===
              Canal_0    Canal_1    Canal_2    Canal_3    Canal_4    Canal_5  \
Sujeto_C_0   7.074337   9.142000  14.374966   7.135363  14.393723   7.330709   
Sujeto_C_1   4.851287   4.155300   5.637089   3.569491   5.143035   8.075745   
Sujeto_C_2  12.903799  15.574167  14.320193  10.512889   9.125902  74.112767   
Sujeto_C_3   9.255415   9.869245  10.455896  10.914127  11.675996  26.543030   
Sujeto_C_4  16.160636  18.851899  21.306986  14.626205  13.445428  34.121809   

              Canal_6    Canal_7  
Sujeto_C_0   7.970077   9.749949  
Sujeto_C_1   7.865648  10.478360  
Sujeto_C_2  61.977140  69.550354  
Sujeto_C_3  18.747986  21.533548  
Sujeto_C_4  26.208961  36.057925  

=== PARKINSON ===
              Canal_0    Canal_1    Canal_2    Canal_3    Canal_4     Canal_5  \
Sujeto_P_0   6.078615   6.699329   8.834439   7.420552   5.648871   19.350824   
Sujeto_P_1  24.461937  32.162385  42.587957  39.043613  32.896313  229.315772   
Sujeto_P_2   8.000803   6.83905

Se observa que el grupo de pacientes con Parkinson presenta valores de energía significativamente mayores en comparación con el grupo control en la mayoría de los canales analizados. Además, el grupo Parkinson muestra una mayor variabilidad entre sujetos, lo que sugiere una actividad neuronal más irregular. Estas diferencias indican que la energía de las señales EEG puede ser un indicador útil para discriminar entre sujetos sanos y pacientes con enfermedad de Parkinson. Sin embargo, es importante considerar posibles diferencias en la escala de las señales, por lo que se recomienda complementar el análisis con pruebas estadísticas.

# **Punto 3**

Determinar si hay diferencias estadísticas entre canales

In [58]:
from scipy.stats import shapiro

print("NORMALIDAD (Shapiro)")

normalidad = []

for col in df_control.columns:
    stat_c, p_c = shapiro(df_control[col])
    stat_p, p_p = shapiro(df_parkinson[col])

    normal = (p_c > 0.05) and (p_p > 0.05)

    normalidad.append((col, p_c, p_p, normal))

    print(f"{col} -> Control p={p_c:.4f}, Parkinson p={p_p:.4f} -> Normal: {normal}")

NORMALIDAD (Shapiro)
Canal_0 -> Control p=0.0063, Parkinson p=0.0144 -> Normal: False
Canal_1 -> Control p=0.0040, Parkinson p=0.0045 -> Normal: False
Canal_2 -> Control p=0.0089, Parkinson p=0.0009 -> Normal: False
Canal_3 -> Control p=0.0002, Parkinson p=0.0003 -> Normal: False
Canal_4 -> Control p=0.0008, Parkinson p=0.0053 -> Normal: False
Canal_5 -> Control p=0.0000, Parkinson p=0.0000 -> Normal: False
Canal_6 -> Control p=0.0000, Parkinson p=0.0000 -> Normal: False
Canal_7 -> Control p=0.0000, Parkinson p=0.0000 -> Normal: False


Los resultados de la prueba de normalidad de Shapiro-Wilk muestran que, para todos los canales evaluados (Canal_0 a Canal_7), los valores de p son menores a 0.05 tanto en el grupo control como en el grupo de pacientes con Parkinson. Esto indica que los datos no siguen una distribución normal en ninguno de los canales, lo cual implica que no se cumple uno de los supuestos fundamentales para la aplicación de pruebas paramétricas como la prueba t de Student.

Adicionalmente, se observa que los valores de p son consistentemente bajos en todos los canales, lo que sugiere una desviación sistemática de la normalidad en las señales EEG analizadas. Este comportamiento ya que suelen presentar alta variabilidad, presencia de ruido y posibles valores atípicos, especialmente en poblaciones clínicas como pacientes con enfermedad de Parkinson.



In [59]:
from scipy.stats import levene

print("HOMOCEDASTICIDAD (Levene)")

homocedasticidad = []

for col in df_control.columns:
    stat, p = levene(df_control[col], df_parkinson[col])

    equal_var = p > 0.05
    homocedasticidad.append((col, p, equal_var))

    print(f"{col} -> p={p:.4f} -> Varianzas iguales: {equal_var}")

HOMOCEDASTICIDAD (Levene)
Canal_0 -> p=0.8848 -> Varianzas iguales: True
Canal_1 -> p=0.9591 -> Varianzas iguales: True
Canal_2 -> p=0.9776 -> Varianzas iguales: True
Canal_3 -> p=0.7593 -> Varianzas iguales: True
Canal_4 -> p=0.9046 -> Varianzas iguales: True
Canal_5 -> p=0.4720 -> Varianzas iguales: True
Canal_6 -> p=0.6179 -> Varianzas iguales: True
Canal_7 -> p=0.5941 -> Varianzas iguales: True


Los resultados de la prueba de homocedasticidad de Levene indican que, para todos los canales analizados, los valores de p son mayores a 0.05. Esto significa que no se rechaza la hipótesis nula de igualdad de varianzas entre el grupo control y el grupo de pacientes con Parkinson, lo que implica que ambos grupos presentan varianzas homogéneas en cada uno de los canales evaluados.

Este resultado sugiere que la dispersión de los datos es similar entre los dos grupos, lo cual es un supuesto importante cuando se consideran pruebas estadísticas paramétricas. Sin embargo, aunque se cumple la homocedasticidad, este no es el único requisito necesario, ya que también se debe cumplir la normalidad de los datos, lo cual, como se observó previamente, no se satisface en este caso.

En consecuencia, a pesar de que las varianzas son iguales entre los grupos, la falta de normalidad impide el uso de pruebas paramétricas como la prueba t. Por ello, es más adecuado emplear una prueba no paramétrica como la U de Mann-Whitney, que no depende de estos supuestos y permite realizar comparaciones válidas entre ambos grupos.

In [60]:
from scipy.stats import ttest_ind, mannwhitneyu

print("PRUEBA FINAL POR CANAL")

resultados = []

for i, col in enumerate(df_control.columns):

    p_norm_c = normalidad[i][1]
    p_norm_p = normalidad[i][2]
    p_levene = homocedasticidad[i][1]

    # Condiciones
    normal = (p_norm_c > 0.05) and (p_norm_p > 0.05)
    equal_var = p_levene > 0.05

    if normal:
        # t-test
        stat, p = ttest_ind(df_control[col], df_parkinson[col], equal_var=equal_var)
        metodo = "t-test"
    else:
        # Mann-Whitney
        stat, p = mannwhitneyu(df_control[col], df_parkinson[col])
        metodo = "Mann-Whitney"

    significativo = p < 0.05

    resultados.append((col, metodo, p, significativo))

    print(f"{col} -> {metodo} | p={p:.5f} | Significativo: {significativo}")

PRUEBA FINAL POR CANAL
Canal_0 -> Mann-Whitney | p=0.40570 | Significativo: False
Canal_1 -> Mann-Whitney | p=0.57053 | Significativo: False
Canal_2 -> Mann-Whitney | p=0.46037 | Significativo: False
Canal_3 -> Mann-Whitney | p=0.23446 | Significativo: False
Canal_4 -> Mann-Whitney | p=0.56002 | Significativo: False
Canal_5 -> Mann-Whitney | p=0.28007 | Significativo: False
Canal_6 -> Mann-Whitney | p=0.11830 | Significativo: False
Canal_7 -> Mann-Whitney | p=0.15054 | Significativo: False


Debido a la falta de normalidad en ambos grupos, no es adecuado aplicar pruebas estadísticas paramétricas para comparar las energías entre el grupo control y el grupo Parkinson. En su lugar, se justifica el uso de pruebas no paramétricas, como la prueba U de Mann-Whitney, la cual no asume normalidad en los datos y es más robusta ante este tipo de distribuciones.

# **Análisis**

Los resultados de la prueba no paramétrica de Mann-Whitney, muestra que todos los valores de p son mayores a 0.05. Esto indica que no existen diferencias estadísticamente significativas entre el grupo control y el grupo de pacientes con Parkinson en ninguno de los canales analizados.

Es importante destacar que, debido a la falta de normalidad en los datos, la elección de la prueba de Mann-Whitney fue adecuada, ya que no requiere supuestos de distribución normal. Sin embargo, incluso utilizando este enfoque más robusto, no se encontraron evidencias suficientes para afirmar que la energía de las señales EEG difiere significativamente entre sujetos sanos y pacientes con enfermedad de Parkinson en los canales evaluados.

En conclusión, aunque existen diferencias aparentes en los valores de energía entre los grupos, estas no resultan estadísticamente significativas según la prueba aplicada. Por lo tanto, bajo las condiciones del análisis realizado, la energía de las señales EEG no permite discriminar de manera concluyente entre el grupo control y el grupo de pacientes con Parkinson.

# **REFERENCIAS**

Facultad de Ciencias Exactas, Ingeniería y Agrimensura. (s.f.). Señales de energía y señales de potencia. Universidad Nacional de Rosario. https://www.fceia.unr.edu.ar/tesys/html/senales_potencia_energia_bw.pdf

Jaramillo-Jiménez, A., Tovar-Ríos, D. A., Ospina, J. A., Mantilla-Ramos, Y. J., Loaiza-López, D., Henao Isaza, V., Zapata Saldarriaga, L. M., Cadavid Castro, V., Suarez-Revelo, J. X., Bocanegra, Y., Lopera, F., Pineda-Salazar, D. A., Tobón Quintero, C. A., Ochoa-Gomez, J. F., Borda, M. G., Aarsland, D., Bonanni, L., & Brønnick, K. (2023). Spectral features of resting-state EEG in Parkinson’s disease: A multicenter study using functional data analysis. Clinical Neurophysiology, 151, 28–40. https://doi.org/10.1016/j.clinph.2023.03.363


Aljalal, M., Aldosari, S. A., AlSharabi, K., Abdurraqeeb, A. M., & Alturki, F. A. (2022). Parkinson’s disease detection from resting-state EEG signals using common spatial pattern, entropy, and machine learning techniques. Diagnostics, 12(5), 1033. https://doi.org/10.3390/diagnostics12051033


Malato, G. (2025, agosto 1). An introduction to the Shapiro-Wilk test for normality. Built In. https://builtin.com/data-science/shapiro-wilk-tes

Levene’s Test: Your Guide to Understanding, Performing, and Interpreting Homogeneity of Variance

Six Sigma. (2024, diciembre 10). Levene’s test: Your guide to understanding, performing, and interpreting homogeneity of variance. https://www.6sigma.us/six-sigma-in-focus/levenes-test/


Narvaez, M. (2022, 15 enero). Prueba U de Mann-Whitney: Qué es y cómo funciona. QuestionPro. https://www.questionpro.com/blog/es/prueba-u-de-mann-whitney/